In [2]:
import pandas as pd
import numpy as np
import requests
import os
import json
from dotenv import load_dotenv
from datetime import datetime, timedelta
import time
import logging

In [3]:
load_dotenv(override = True)

True

In [4]:
api_key = os.getenv('API_TOKEN')
api_url = os.getenv('API_URL')


In [5]:
try:
    data = {
        'token': api_key,
        'content': 'record',
        'action': 'export',
        'format': 'json',
        'type': 'flat',
        'csvDelimiter': '',
        'fields[0]': 'record_id',
        'forms[0]': 'dados_pessoais',
        'events[0]': 'vsv1_arm_1',
        'rawOrLabel': 'label',
        'rawOrLabelHeaders': 'raw',
        'exportCheckboxLabel': 'false',
        'exportSurveyFields': 'false',
        'exportDataAccessGroups': 'false',
        'returnFormat': 'json'
    }
    response = requests.post(f'{api_url}', data=data)
    response.raise_for_status()
    records = response.json()
    df = pd.DataFrame(records)
    record_id_list = df['record_id'].tolist()

except requests.exceptions.RequestException as e:
    logging.error(f"An error occurred while making the API request: {e}")
except json.JSONDecodeError as e:
    logging.error(f"An error occurred while decoding the JSON response: {e}")
except KeyError as e:
    logging.error(f"Key error: {e}")
except Exception as e:
    logging.error(f"An unexpected error occurred: {e}")


In [ ]:
events_data = {
    'token': 'AF55337FBFB7272EAB787406E11DFCD7',
    'content': 'event',
    'format': 'json',
    'arms[0]': '1',
    'returnFormat': 'json'
}
response = requests.post('https://svriglobal.org/redcap/api/',data=events_data)
print('HTTP Status: ' + str(response.status_code))
record = response.json()
events_names_df = pd.DataFrame(record)
events_names_df

HTTP Status: 200


,event_name,arm_num,day_offset,offset_min,offset_max,unique_event_name,custom_event_label,event_id
0,Status,1,0,0,0,status_arm_1,None,347
1,VS/V1,1,1,5,0,vsv1_arm_1,None,340
2,VR/V2,1,2,4,1,vrv2_arm_1,None,341
3,V3/VF,1,3,21,30,v3vf_arm_1,None,342
4,Visita não programada,1,4,0,0,visita_no_programa_arm_1,None,343
5,Eventos Adversos,1,5,0,0,eventos_adversos_arm_1,None,344
6,Medicações Concomitantes,1,5,0,0,medicaes_concomita_arm_1,None,345
7,Reconsentimento,1,5,0,0,reconsentimento_arm_1,None,346
8,Diários Dia 1,1,6,0,0,dirios_dia_1_arm_1,None,348
9,Diários Dia 2,1,7,0,0,dirios_dia_2_arm_1,None,349


In [12]:
#Isolando as visitas que contêm a palavra Diários como parte do nome em event_name
diarios_events_df = events_names_df[events_names_df['event_name'].str.contains('Diários')]
diarios_list = diarios_events_df['event_name'].tolist()
print(diarios_list)

['Diários Dia 1', 'Diários Dia 2', 'Diários Dia 3', 'Diários Dia 4', 'Diários Dia 5', 'Diários Dia 6', 'Diários Dia 7', 'Diários Dia 8', 'Diários Dia 9', 'Diários Dia 10', 'Diários Dia 11', 'Diários Dia 12', 'Diários Dia 13', 'Diários Dia 14', 'Diários Dia 15', 'Diários Dia 16', 'Diários Dia 17', 'Diários Dia 18', 'Diários Dia 19', 'Diários Dia 20', 'Diários Dia 21', 'Diários Dia 22', 'Diários Dia 23', 'Diários Dia 24', 'Diários Dia 25', 'Diários Dia 26', 'Diários Dia 27', 'Diários Dia 28']


In [13]:
# Obtendo o grupo ao qual o participante pertence

for record_id in record_id_list:
    try: 
        randomization_data = {
            'token': api_key,
            'content': 'record',
            'action': 'export',
            'format': 'json',
            'type': 'flat',
            'csvDelimiter': '',
            'records[0]': record_id,
            'fields[0]': 'record_id',
            'fields[1]': 'randomizacao_q3',
            'events[0]': 'vrv2_arm_1',
            'rawOrLabel': 'raw',
            'rawOrLabelHeaders': 'raw',
            'exportCheckboxLabel': 'false',
            'exportSurveyFields': 'false',
            'exportDataAccessGroups': 'false',
            'returnFormat': 'json'
        }
        response = requests.post(f'{api_url}', data=randomization_data)
        response.raise_for_status()
        randomization_record = response.json()
        randomization_df = pd.DataFrame(randomization_record)
        group = randomization_df['randomizacao_q3'].iloc[0]
        print(f"Record ID: {record_id}, Group: {group}")
    except requests.exceptions.RequestException as e:
        logging.error(f"An error occurred while making the API request for record {record_id}: {e}")
    except json.JSONDecodeError as e:        
        logging.error(f"An error occurred while decoding the JSON response for record {record_id}: {e}")
    except KeyError as e:
        logging.error(f"Key error for record {record_id}: {e}")
    except Exception as e:
        logging.error(f"An unexpected error occurred for record {record_id}: {e}")

Record ID: 93-1, Group: 2
Record ID: 93-2, Group: 3
Record ID: 93-3, Group: 1
Record ID: 93-4, Group: 2
Record ID: 93-5, Group: 3
Record ID: 93-6, Group: 1
Record ID: 93-7, Group: 2
Record ID: 93-8, Group: 1
Record ID: 93-9, Group: 3
Record ID: 93-10, Group: 1
Record ID: 93-11, Group: 3
Record ID: 93-12, Group: 2
Record ID: 93-13, Group: 3
Record ID: 93-14, Group: 2
Record ID: 93-15, Group: 1
Record ID: 93-17, Group: 2
Record ID: 93-18, Group: 1
Record ID: 93-19, Group: 3
Record ID: 93-21, Group: 1
Record ID: 93-22, Group: 1
Record ID: 93-26, Group: 1
Record ID: 93-30, Group: 1
Record ID: 93-32, Group: 2
Record ID: 93-33, Group: 3
Record ID: 93-34, Group: 1
Record ID: 93-35, Group: 3
Record ID: 93-36, Group: 1
Record ID: 93-37, Group: 2
Record ID: 93-38, Group: 2
Record ID: 93-39, Group: 1
Record ID: 93-40, Group: 3
Record ID: 93-41, Group: 3
Record ID: 93-42, Group: 1
Record ID: 93-43, Group: 2
Record ID: 93-44, Group: 3
Record ID: 93-45, Group: 2
Record ID: 93-46, Group: 1
Record ID:

In [ ]:
# DE-PARA entre group e o nome do instrumento de coleta de dados
if group == '1':
    instrument_name = "diario_serum_ultra_repositor"
elif group == '2':
    instrument_name = "diario_hidratante_ultra_refrescante"
elif group == '3':
    instrument_name = "dairio_tratamento_intensivo_noturno"
else:
    logging.error(f"Unexpected group value: {group}")
    instrument_name = None
